# LGBM Regression Training - Spearman First

Train deployable LightGBM regression models for the long, short, and signed quality targets.
The notebook uses purge-aware walk-forward validation, tunes by Spearman, and exports a model pack
that can be consumed by the production stack.

In [25]:
import inspect
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy.stats as stats
import lightgbm as lgb
from IPython.display import display
from lightgbm import LGBMRegressor, LGBMClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, average_precision_score

from Learn.features import add_feature_library, _add_features_US500_v2, select_features_pipeline
from Learn.labels import calculate_trade_outcomes_capped, generate_tradeability_scores
from Learn.preprocess import preprocess_ohlcv

warnings.filterwarnings("ignore")


def load_ohlcv(ds_name, n_rows=None):
    """Load and prepare OHLCV data from a CSV file."""
    df = pd.read_csv(ds_name)
    if n_rows is not None:
        df = df.tail(n_rows)
    df = df.sort_values("Time").reset_index(drop=True)
    df["Time"] = pd.to_datetime(df["Time"])
    return df


def safe_spearman(y_true, y_pred):
    """Compute Spearman rank correlation with safe fallback to 0.0."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if len(y_true) < 2 or np.nanstd(y_true) == 0.0 or np.nanstd(y_pred) == 0.0:
        return 0.0
    value = stats.spearmanr(y_true, y_pred, nan_policy="omit").statistic
    return 0.0 if value is None or np.isnan(value) else float(value)


def reg_metrics(y_true, y_pred):
    """Compute a dict of standard regression metrics."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": r2_score(y_true, y_pred),
        "Spearman": safe_spearman(y_true, y_pred),
    }


def lgbm_spearman_eval(y_true, y_pred):
    """Custom LightGBM eval function tracking Spearman (higher is better)."""
    return "spearman", safe_spearman(y_true, y_pred), True


def walk_forward_folds(n_samples, min_train_size, test_size, gap_size, n_folds):
    """
    Generate expanding-window walk-forward train/val indices.

    Each fold uses all data up to train_end for training, then
    a contiguous test block separated by a gap.
    """
    folds = []
    train_end = int(min_train_size)
    for _ in range(int(n_folds)):
        test_start = train_end + int(gap_size)
        if test_start >= n_samples:
            break
        test_end = min(test_start + int(test_size), n_samples)
        if test_end <= test_start:
            break
        train_idx = np.arange(0, train_end, dtype=np.int64)
        test_idx = np.arange(test_start, test_end, dtype=np.int64)
        if len(train_idx) == 0 or len(test_idx) == 0:
            break
        folds.append((train_idx, test_idx))
        train_end = test_end
        if train_end >= n_samples:
            break
    return folds


def describe_target_frame(frame, target_cols):
    """Return a summary DataFrame of target distributions."""
    rows = []
    for col in target_cols:
        s = pd.Series(frame[col])
        rows.append({
            "target": col,
            "mean": float(s.mean()),
            "std": float(s.std()),
            "p01": float(s.quantile(0.01)),
            "p50": float(s.quantile(0.50)),
            "p99": float(s.quantile(0.99)),
        })
    return pd.DataFrame(rows).set_index("target")


def tune_lgbm_by_spearman(
    X,
    y,
    folds,
    candidate_params,
    base_params,
    sample_weight=None,
    early_stopping_rounds=200,
    robustness_penalty=0.25,
):
    """
    Tune hyper-parameter candidates via walk-forward cross-validation.

    Ranks candidates by a robust score = mean Spearman - penalty * std Spearman,
    then mean Spearman, then mean R². Returns (results_df, best_cfg_dict).
    """
    rows = []
    for cfg_i, cfg in enumerate(candidate_params, start=1):
        fold_rows = []
        best_iters = []
        for tr_idx, va_idx in folds:
            model = LGBMRegressor(**{**base_params, **cfg})
            fit_kwargs = {
                "X": X[tr_idx],
                "y": y[tr_idx],
                "eval_set": [(X[va_idx], y[va_idx])],
                "eval_metric": lgbm_spearman_eval,
                "callbacks": [lgb.early_stopping(early_stopping_rounds, verbose=False)],
            }
            if sample_weight is not None:
                fit_kwargs["sample_weight"] = sample_weight[tr_idx]
            model.fit(**fit_kwargs)

            best_iter = getattr(model, "best_iteration_", None)
            if best_iter is None or best_iter <= 0:
                best_iter = model.n_estimators
            best_iters.append(int(best_iter))

            pred = model.predict(X[va_idx], num_iteration=best_iter)
            fold_rows.append(reg_metrics(y[va_idx], pred))

        fold_df = pd.DataFrame(fold_rows)
        rows.append({
            **cfg,
            "cfg_id": cfg_i,
            "folds": len(fold_rows),
            "mean_spearman": float(fold_df["Spearman"].mean()),
            "std_spearman": float(fold_df["Spearman"].std(ddof=0)),
            "mean_r2": float(fold_df["R2"].mean()),
            "mean_rmse": float(fold_df["RMSE"].mean()),
            "mean_mae": float(fold_df["MAE"].mean()),
            "robust_score": float(
                fold_df["Spearman"].mean()
                - robustness_penalty * fold_df["Spearman"].std(ddof=0)
            ),
            "median_best_iteration": int(np.median(best_iters)),
            "fold_metrics": fold_rows,
        })

    results = pd.DataFrame(rows).sort_values(
        ["robust_score", "mean_spearman", "mean_r2"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    best = results.iloc[0].to_dict()
    return results, best

In [26]:
# ========== CORE CONFIGURATION ==========
DS_NAME = "../data/US500_M1_520weeks.csv"
N_ROWS = 500_000

# --- Asset-specific settings ---
TP_MULT = 4.0
SL_MULT = 2.5
TARGET_CLIP_MAX = max(TP_MULT, SL_MULT) * 1.5
TRAIN_FRACTION = 0.90

# --- Walk-forward validation ---
MAX_TUNING_ROWS = 350_000
WFO_FOLDS = 4
WFO_MIN_TRAIN_FRAC = 0.55
WFO_TEST_FRAC = 0.10
WFO_GAP_ROWS = max(60, 30 * 2)  # At least 60 bars purge gap
EARLY_STOPPING_ROUNDS = 200
ROBUSTNESS_PENALTY = 0.25

# --- Feature engineering ---
USE_FEATURE_SELECTION = True
SELECTED_FEATURE_COUNT = 80

# --- Tradeability weighting ---
USE_TRADEABILITY_WEIGHTING = True
TRADEABILITY_N_FOLDS = 5
TRADEABILITY_MIN_TRAIN = 100_000
TRADEABILITY_TEST_SIZE = 50_000

# --- Targets ---
TARGET_COLS = ["long_quality", "short_quality", "signed_quality"]
CLASSIFICATION_TARGETS = ["buy_win", "sell_win", "signed_win"]
PRIMARY_TARGET = "signed_quality"
TIME_PENALTY_LAMBDA = 0.1

# --- Optional model enhancements ---
TRAIN_CLASSIFIER = True
ENSEMBLE_SIZE = 5
ENSEMBLE_SEEDS = [42, 123, 456, 789, 1111]

# --- Regime params ---
regime_params = {
    "ma_period": 90,
    "slope_smoothness": 50,
    "regime_min_duration": 0,
    "atr_window": 60,
    "atr_lookback": 720,
    "atr_percentile": 0.0,
    "slope_threshold": 0,
}

# --- Outcome params (must match backtest) ---
outcome_params = {
    "atr_window": 60,
    "tp_mult": TP_MULT,
    "sl_mult": SL_MULT,
    "max_horizon": 30,
}

# --- LGBM base params ---
LGBM_BASE_PARAMS = {
    "objective": "regression",
    "boosting_type": "gbdt",
    "n_estimators": 5000,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
    "max_depth": -1,
    "subsample_freq": 1,
}

LGBM_CANDIDATES = [
    dict(learning_rate=0.03, num_leaves=63, min_child_samples=60, subsample=0.80, colsample_bytree=0.80, reg_alpha=0.10, reg_lambda=0.50, min_split_gain=0.00),
    dict(learning_rate=0.02, num_leaves=127, min_child_samples=80, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.20, reg_lambda=1.00, min_split_gain=0.00),
    dict(learning_rate=0.01, num_leaves=255, min_child_samples=120, subsample=0.90, colsample_bytree=0.90, reg_alpha=0.50, reg_lambda=3.00, min_split_gain=0.00),
    dict(learning_rate=0.05, num_leaves=31, min_child_samples=100, subsample=0.75, colsample_bytree=0.75, reg_alpha=0.05, reg_lambda=5.00, min_split_gain=0.00),
    dict(learning_rate=0.025, num_leaves=63, min_child_samples=40, subsample=0.85, colsample_bytree=0.80, reg_alpha=0.10, reg_lambda=2.00, min_split_gain=0.01),
    dict(learning_rate=0.02, num_leaves=95, min_child_samples=50, subsample=0.80, colsample_bytree=0.75, reg_alpha=0.00, reg_lambda=1.00, min_split_gain=0.00),
]

# Select feature function based on whether we use automated selection
if USE_FEATURE_SELECTION:
    def FEATURES(df, include_mtf=True, regime_params=None):
        return _add_features_US500_v2(
            df, include_mtf=include_mtf, regime_params=regime_params
        )
else:
    FEATURES = _add_features_US500

print("Device: CPU (tree model)")
print("Dataset:", DS_NAME)
print("Targets:", TARGET_COLS)
print("Feature function:", getattr(FEATURES, "__name__", str(FEATURES)))
print(f"R:R = {TP_MULT/SL_MULT:.2f} (TP_MULT={TP_MULT}, SL_MULT={SL_MULT})")
print(f"Purge gap: {WFO_GAP_ROWS} bars")

Device: CPU (tree model)
Dataset: ../data/US500_M1_520weeks.csv
Targets: ['long_quality', 'short_quality', 'signed_quality']
Feature function: FEATURES
R:R = 1.60 (TP_MULT=4.0, SL_MULT=2.5)
Purge gap: 60 bars


In [27]:
# Load data and compute target variables
from talib import ATR

df = load_ohlcv(DS_NAME, n_rows=N_ROWS)
print(f"Loaded {len(df):,} rows")
print(df["Time"].min(), "->", df["Time"].max())

# Step 1: Generate features
df = FEATURES(df.copy(), include_mtf=True, regime_params=regime_params)
print(f'Added features: {len(df.columns)} columns')

# Step 2: Feature selection (if enabled)
if USE_FEATURE_SELECTION:
    print("Running automated feature selection...")
    # Compute targets first on the full feature set for selection
    df_for_selection = df.copy()
    df_for_selection["atr"] = ATR(df_for_selection["High"], df_for_selection["Low"], df_for_selection["Close"], timeperiod=14)
    outcomes_fs = calculate_trade_outcomes_capped(df_for_selection, **outcome_params)
    eps_fs = 1e-8
    long_q_fs = np.log1p(outcomes_fs["buy_MFE"] / (outcomes_fs["buy_MAE"] + eps_fs))
    short_q_fs = np.log1p(outcomes_fs["sell_MFE"] / (outcomes_fs["sell_MAE"] + eps_fs))
    if TARGET_CLIP_MAX is not None:
        long_q_fs = np.clip(long_q_fs, 0.0, TARGET_CLIP_MAX)
        short_q_fs = np.clip(short_q_fs, 0.0, TARGET_CLIP_MAX)
    df_for_selection["signed_quality"] = long_q_fs - short_q_fs
    
    # Use a subset for speed (last 500k rows)
    fs_subset = df_for_selection.tail(min(500_000, len(df_for_selection)))
    selected_features = select_features_pipeline(
        fs_subset,
        target_col='signed_quality',
        n_features=SELECTED_FEATURE_COUNT,
        correlation_threshold=0.95,
        include_mtf=True,
        regime_params=regime_params,
    )
    print(f"Selected {len(selected_features)} features")
    
    # Re-generate features from FRESH raw data with only selected columns.
    # IMPORTANT: Use the original raw df (before any feature engineering) to avoid
    # _x/_y suffixed duplicate columns that occur when _add_features_US500_v2 is
    # called on a dataframe that already has MTF columns.
    df_raw_fresh = load_ohlcv(DS_NAME, n_rows=N_ROWS)
    df = _add_features_US500_v2(
        df_raw_fresh, include_mtf=True, regime_params=regime_params,
        selected_features=selected_features
    )
    print(f"Feature matrix: {len(df)} rows x {len(selected_features)} features")
    
    # Update FEATURES for model pack export (single-pass, uses selected_features)
    def FEATURES(df, include_mtf=True, regime_params=None):
        return _add_features_US500_v2(
            df, include_mtf=include_mtf, regime_params=regime_params,
            selected_features=selected_features
        )
else:
    # Still need ATR for non-selection path
    df["atr"] = ATR(df["High"], df["Low"], df["Close"], timeperiod=14)
    print('Added ATR')

# Step 3: Compute ATR (needed for label computation)
df["atr"] = ATR(df["High"], df["Low"], df["Close"], timeperiod=14)

# Step 4: Compute trade outcomes using horizon-capped label function
outcomes = calculate_trade_outcomes_capped(df, **outcome_params)
print(f"Calculated trade outcomes for {len(outcomes):,} rows")

eps = 1e-8
long_q = np.log1p(outcomes["buy_MFE"] / (outcomes["buy_MAE"] + eps))
short_q = np.log1p(outcomes["sell_MFE"] / (outcomes["sell_MAE"] + eps))
print(f"Computed long and short quality metrics for {len(long_q):,} rows")

if TARGET_CLIP_MAX is not None:
    long_q = np.clip(long_q, 0.0, TARGET_CLIP_MAX)
    short_q = np.clip(short_q, 0.0, TARGET_CLIP_MAX)
print(f"Clipped long and short quality metrics to max {TARGET_CLIP_MAX}")

# Time penalty for slow trades
bars_to_exit_buy = np.full(len(outcomes), np.nan)
bars_to_exit_sell = np.full(len(outcomes), np.nan)
for i in range(len(outcomes)):
    for j in range(i + 1, min(i + outcome_params['max_horizon'] + 1, len(outcomes))):
        if not np.isnan(outcomes['buy_outcome'].iloc[j]):
            bars_to_exit_buy[i] = j - i
            break
    for j in range(i + 1, min(i + outcome_params['max_horizon'] + 1, len(outcomes))):
        if not np.isnan(outcomes['sell_outcome'].iloc[j]):
            bars_to_exit_sell[i] = j - i
            break

buy_time_penalty = np.exp(-TIME_PENALTY_LAMBDA * bars_to_exit_buy / outcome_params['max_horizon'])
sell_time_penalty = np.exp(-TIME_PENALTY_LAMBDA * bars_to_exit_sell / outcome_params['max_horizon'])
buy_time_penalty = np.nan_to_num(buy_time_penalty, nan=0.0)
sell_time_penalty = np.nan_to_num(sell_time_penalty, nan=0.0)
long_q = long_q * buy_time_penalty
short_q = short_q * sell_time_penalty
print(f"Applied time penalty with lambda={TIME_PENALTY_LAMBDA}")

df["long_quality"] = long_q
df["short_quality"] = short_q
df["signed_quality"] = df["long_quality"] - df["short_quality"]

# Binary classification targets
df['buy_win'] = (outcomes['buy_outcome'] == 1.0).astype(float)
df['sell_win'] = (outcomes['sell_outcome'] == 1.0).astype(float)
df['signed_win'] = np.where(
    df['signed_quality'] > 0, df['buy_win'],
    np.where(df['signed_quality'] < 0, df['sell_win'], 0.0)
)

print("Target summary:")
display(describe_target_frame(df, TARGET_COLS))

print(
    f"Zero share | long={(df['long_quality'] <= 1e-12).mean():.3f} "
    f"short={(df['short_quality'] <= 1e-12).mean():.3f} "
    f"signed={(df['signed_quality'] == 0).mean():.3f}"
)

Loaded 500,000 rows
2025-01-16 21:43:00+00:00 -> 2026-06-19 16:59:00+00:00
Added features: 203 columns
Running automated feature selection...
Selected 80 features
Feature matrix: 500000 rows x 80 features
Calculated trade outcomes for 500,000 rows
Computed long and short quality metrics for 500,000 rows
Clipped long and short quality metrics to max 6.0
Applied time penalty with lambda=0.1
Target summary:


,mean,std,p01,p50,p99
target,,,,,
long_quality,0.576312,0.860450,0.000000,0.296503,5.960133
short_quality,0.573879,0.846945,0.000000,0.302221,5.960133
signed_quality,0.002434,1.412816,-5.960133,0.000000,5.952596


Zero share | long=0.250 short=0.247 signed=0.014


In [28]:
# Generate Tradeability Scores
if USE_TRADEABILITY_WEIGHTING:
    print("Generating tradeability scores...")
    tradeability_y = pd.Series(
        (~outcomes['buy_outcome'].isna() | ~outcomes['sell_outcome'].isna()).astype(int),
        name='tradeable'
    )
    
    # Use top features by variance for the tradeability classifier
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    exclude_cols = set(TARGET_COLS + CLASSIFICATION_TARGETS + ['tradeability_score', 'atr'])
    feature_candidates = [c for c in numeric_cols if c not in exclude_cols]
    
    if len(feature_candidates) > 50:
        variances = df[feature_candidates].var().sort_values(ascending=False)
        top_features = variances.head(50).index.tolist()
    else:
        top_features = feature_candidates
    
    tradeability_scores, tradeability_metrics = generate_tradeability_scores(
        X=df[top_features].values,
        y=tradeability_y.values,
        timestamps=df['Time'],
        n_folds=TRADEABILITY_N_FOLDS,
        min_train_size=TRADEABILITY_MIN_TRAIN,
        test_size=TRADEABILITY_TEST_SIZE,
        gap_size=outcome_params['max_horizon'],
    )
    
    df['tradeability_score'] = tradeability_scores['tradeability_score'].values
    # Rows never assigned to a test fold will have NaN; default to neutral 0.5
    df['tradeability_score'] = df['tradeability_score'].fillna(0.5)
    print(f"Tradeability score range: [{df['tradeability_score'].min():.4f}, {df['tradeability_score'].max():.4f}]")
    print(f"Tradeability ROC AUC: {tradeability_metrics['overall_roc_auc']:.4f}")
    print(f"Tradeability PR AUC: {tradeability_metrics['overall_pr_auc']:.4f}")
else:
    df['tradeability_score'] = 1.0

Generating tradeability scores...
Processing fold 1/5: train [0:99999], test [100030:150029]
Processing fold 2/5: train [0:150029], test [150060:200059]
Processing fold 3/5: train [0:200059], test [200090:250089]
Processing fold 4/5: train [0:250089], test [250120:300119]
Processing fold 5/5: train [0:300119], test [300150:350149]
Tradeability score range: [0.0090, 1.0000]
Tradeability ROC AUC: 0.7685
Tradeability PR AUC: 0.9728


In [29]:
# Preprocess features and create train / holdout splits
preprocess_ohlcv_args = {
    "target_col": TARGET_COLS + ["tradeability_score"] + (CLASSIFICATION_TARGETS if TRAIN_CLASSIFIER else []),
    "outcomes_col": None,
    "shift": 0,
    "onehot_prefixes": ["OH_"],
    "price_prefixes": ["PR_"],
}

split_idx = int(len(df) * TRAIN_FRACTION)
df_train = df.iloc[:split_idx].copy().reset_index(drop=True)
df_holdout = df.iloc[split_idx:].copy().reset_index(drop=True)

X_train, y_train, scaler, features, _, proc_df_train = preprocess_ohlcv(
    df_train, **preprocess_ohlcv_args, scaler=None, return_df=True,
)
X_holdout, y_holdout, _, _, _, proc_df_holdout = preprocess_ohlcv(
    df_holdout, **preprocess_ohlcv_args, scaler=scaler, return_df=True,
)

# Split target arrays for regression targets
regression_target_count = len(TARGET_COLS)
target_arrays = {
    target: {
        "train": y_train[:, i],
        "holdout": y_holdout[:, i],
    }
    for i, target in enumerate(TARGET_COLS)
}

# Classification target arrays
if TRAIN_CLASSIFIER:
    clf_target_arrays = {
        target: {
            "train": y_train[:, regression_target_count + 1 + i],
            "holdout": y_holdout[:, regression_target_count + 1 + i],
        }
        for i, target in enumerate(CLASSIFICATION_TARGETS)
    }

# Tradeability weight is at index len(TARGET_COLS) (the tradeability_score column)
tradeability_idx = len(TARGET_COLS)
train_weight = None if not USE_TRADEABILITY_WEIGHTING else y_train[:, tradeability_idx]

tune_rows = min(len(X_train), MAX_TUNING_ROWS)
X_tune = X_train[-tune_rows:]
tune_targets = {k: v["train"][-tune_rows:] for k, v in target_arrays.items()}
tune_weight = None if train_weight is None else train_weight[-tune_rows:]

tune_test_size = max(20_000, int(tune_rows * WFO_TEST_FRAC))
tune_min_train = max(100_000, int(tune_rows * WFO_MIN_TRAIN_FRAC))
tune_min_train = min(tune_min_train, tune_rows - tune_test_size - WFO_GAP_ROWS)
while tune_min_train + tune_test_size + WFO_GAP_ROWS > tune_rows and tune_test_size > 10_000:
    tune_test_size = int(tune_test_size * 0.8)
    tune_min_train = min(tune_min_train, tune_rows - tune_test_size - WFO_GAP_ROWS)

wf_folds = walk_forward_folds(
    n_samples=tune_rows,
    min_train_size=tune_min_train,
    test_size=tune_test_size,
    gap_size=max(WFO_GAP_ROWS, outcome_params["max_horizon"]),
    n_folds=WFO_FOLDS,
)

print(f"Train rows   : {len(X_train):,}")
print(f"Holdout rows : {len(X_holdout):,}")
print(f"Tuning rows  : {len(X_tune):,}")
print(f"Walk-forward folds: {len(wf_folds)}")
for i, (tr, va) in enumerate(wf_folds, start=1):
    print(f"  fold {i}: train={len(tr):,} val={len(va):,}")

Train rows   : 449,745
Holdout rows : 49,999
Tuning rows  : 350,000
Walk-forward folds: 4
  fold 1: train=192,500 val=35,000
  fold 2: train=227,560 val=35,000
  fold 3: train=262,620 val=35,000
  fold 4: train=297,680 val=35,000


In [30]:
# Tune hyper-parameters for each target
tuning_results = {}
best_config_by_target = {}
best_iteration_by_target = {}

for target_name in TARGET_COLS:
    print(f"\nTuning target: {target_name}")
    results_df, best_cfg = tune_lgbm_by_spearman(
        X=X_tune,
        y=tune_targets[target_name],
        folds=wf_folds,
        candidate_params=LGBM_CANDIDATES,
        base_params=LGBM_BASE_PARAMS,
        sample_weight=tune_weight,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        robustness_penalty=ROBUSTNESS_PENALTY,
    )
    tuning_results[target_name] = results_df
    best_config_by_target[target_name] = best_cfg
    best_iteration_by_target[target_name] = int(best_cfg["median_best_iteration"])

    print("Top candidates:")
    display(results_df[[
        "cfg_id", "mean_spearman", "std_spearman", "robust_score",
        "mean_r2", "mean_rmse", "median_best_iteration",
        "learning_rate", "num_leaves", "min_child_samples",
        "subsample", "colsample_bytree", "reg_alpha", "reg_lambda",
    ]].head(5).round(4))

tuning_summary = pd.DataFrame([
    {
        "target": target,
        "best_cfg_id": best_config_by_target[target]["cfg_id"],
        "best_mean_spearman": best_config_by_target[target]["mean_spearman"],
        "best_std_spearman": best_config_by_target[target]["std_spearman"],
        "best_robust_score": best_config_by_target[target]["robust_score"],
        "best_n_estimators": best_iteration_by_target[target],
    }
    for target in TARGET_COLS
]).set_index("target")

print("\nSelected tuning summary:")
display(tuning_summary.round(4))


Tuning target: long_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,2,0.2861,0.0076,0.2842,0.0627,0.7783,108,0.020,127,80,0.85,0.85,0.20,1.0
1,4,0.2861,0.0078,0.2841,0.0654,0.7772,57,0.050,31,100,0.75,0.75,0.05,5.0
2,5,0.2858,0.0075,0.2839,0.0642,0.7777,117,0.025,63,40,0.85,0.80,0.10,2.0
3,6,0.2853,0.0068,0.2836,0.0637,0.7779,117,0.020,95,50,0.80,0.75,0.00,1.0
4,1,0.2852,0.0065,0.2836,0.0641,0.7778,76,0.030,63,60,0.80,0.80,0.10,0.5



Tuning target: short_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,4,0.2815,0.0043,0.2805,0.0638,0.7715,70,0.050,31,100,0.75,0.75,0.05,5.0
1,1,0.2813,0.0040,0.2803,0.0580,0.7742,100,0.030,63,60,0.80,0.80,0.10,0.5
2,6,0.2812,0.0044,0.2801,0.0585,0.7740,155,0.020,95,50,0.80,0.75,0.00,1.0
3,2,0.2810,0.0039,0.2800,0.0625,0.7721,130,0.020,127,80,0.85,0.85,0.20,1.0
4,5,0.2806,0.0040,0.2796,0.0631,0.7718,108,0.025,63,40,0.85,0.80,0.10,2.0



Tuning target: signed_quality
Top candidates:


,cfg_id,mean_spearman,std_spearman,robust_score,mean_r2,mean_rmse,median_best_iteration,learning_rate,num_leaves,min_child_samples,subsample,colsample_bytree,reg_alpha,reg_lambda
0,1,0.2313,0.0102,0.2288,0.0695,1.2893,93,0.030,63,60,0.80,0.80,0.10,0.5
1,6,0.2311,0.0099,0.2286,0.0639,1.2929,101,0.020,95,50,0.80,0.75,0.00,1.0
2,4,0.2307,0.0094,0.2284,0.0673,1.2909,48,0.050,31,100,0.75,0.75,0.05,5.0
3,5,0.2307,0.0102,0.2281,0.0685,1.2900,78,0.025,63,40,0.85,0.80,0.10,2.0
4,2,0.2304,0.0098,0.2279,0.0652,1.2926,127,0.020,127,80,0.85,0.85,0.20,1.0



Selected tuning summary:


,best_cfg_id,best_mean_spearman,best_std_spearman,best_robust_score,best_n_estimators
target,,,,,
long_quality,2,0.2861,0.0076,0.2842,108
short_quality,4,0.2815,0.0043,0.2805,70
signed_quality,1,0.2313,0.0102,0.2288,93


In [31]:
# Train final models on full training set and evaluate on holdout
final_models = {}
holdout_results = {}

for target_name in TARGET_COLS:
    best_cfg = best_config_by_target[target_name]
    final_params = {
        **LGBM_BASE_PARAMS,
        **{k: v for k, v in best_cfg.items() if k in LGBM_CANDIDATES[0]},
        "n_estimators": int(best_iteration_by_target[target_name]),
    }

    print(f"Training final model for {target_name} with {final_params['n_estimators']} trees")
    model = LGBMRegressor(**final_params)
    fit_kwargs = {"X": X_train, "y": target_arrays[target_name]["train"]}
    if train_weight is not None:
        fit_kwargs["sample_weight"] = train_weight
    model.fit(**fit_kwargs)

    pred_holdout = model.predict(X_holdout)
    final_models[target_name] = model
    holdout_results[target_name] = reg_metrics(
        target_arrays[target_name]["holdout"], pred_holdout
    )
    holdout_results[target_name]["best_n_estimators"] = int(final_params["n_estimators"])

signed_pred = final_models["signed_quality"].predict(X_holdout)
long_pred = final_models["long_quality"].predict(X_holdout)
short_pred = final_models["short_quality"].predict(X_holdout)
spread_pred = long_pred - short_pred

holdout_results["spread_proxy"] = reg_metrics(
    target_arrays["signed_quality"]["holdout"], spread_pred
)
holdout_results["spread_proxy"]["best_n_estimators"] = int(
    np.median([
        best_iteration_by_target["long_quality"],
        best_iteration_by_target["short_quality"],
    ])
)

holdout_df = pd.DataFrame(holdout_results).T
print("\nHoldout performance:")
display(holdout_df.round(4))

cv_best = pd.DataFrame([
    {
        "target": target,
        "cv_mean_spearman": tuning_summary.loc[target, "best_mean_spearman"],
        "cv_std_spearman": tuning_summary.loc[target, "best_std_spearman"],
        "cv_robust_score": tuning_summary.loc[target, "best_robust_score"],
        "holdout_spearman": holdout_results[target]["Spearman"],
        "holdout_r2": holdout_results[target]["R2"],
    }
    for target in TARGET_COLS
]).set_index("target")

print("\nCV vs holdout:")
display(cv_best.round(4))

Training final model for long_quality with 108 trees
Training final model for short_quality with 70 trees
Training final model for signed_quality with 93 trees

Holdout performance:


,MAE,RMSE,R2,Spearman,best_n_estimators
long_quality,0.5012,0.7321,0.0686,0.2925,108.0
short_quality,0.4936,0.7227,0.0633,0.2849,70.0
signed_quality,0.8966,1.2239,0.0707,0.2327,93.0
spread_proxy,0.8959,1.2238,0.0707,0.2335,89.0



CV vs holdout:


,cv_mean_spearman,cv_std_spearman,cv_robust_score,holdout_spearman,holdout_r2
target,,,,,
long_quality,0.2861,0.0076,0.2842,0.2925,0.0686
short_quality,0.2815,0.0043,0.2805,0.2849,0.0633
signed_quality,0.2313,0.0102,0.2288,0.2327,0.0707


In [32]:
# Train Classification Models
if TRAIN_CLASSIFIER:
    final_classifiers = {}
    classifier_holdout_results = {}
    
    for target_name in CLASSIFICATION_TARGETS:
        print(f"Training classifier for {target_name}")
        
        clf = LGBMClassifier(
            objective='binary',
            boosting_type='gbdt',
            n_estimators=1000,
            learning_rate=0.03,
            num_leaves=63,
            min_child_samples=100,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
            class_weight='balanced',
        )
        
        clf.fit(
            X_train,
            clf_target_arrays[target_name]['train'],
            sample_weight=train_weight,
        )
        
        pred_proba = clf.predict_proba(X_holdout)[:, 1]
        y_true_clf = clf_target_arrays[target_name]['holdout']
        
        classifier_holdout_results[target_name] = {
            'ROC_AUC': float(roc_auc_score(y_true_clf, pred_proba)),
            'PR_AUC': float(average_precision_score(y_true_clf, pred_proba)),
        }
        final_classifiers[target_name] = clf
    
    print("\nClassifier holdout performance:")
    display(pd.DataFrame(classifier_holdout_results).T.round(4))

Training classifier for buy_win
Training classifier for sell_win
Training classifier for signed_win

Classifier holdout performance:


,ROC_AUC,PR_AUC
buy_win,0.6181,0.2552
sell_win,0.5876,0.2531
signed_win,0.6100,0.4919


In [33]:
# Train Ensemble Models
if ENSEMBLE_SIZE > 1:
    ensemble_models = {target: [] for target in TARGET_COLS}
    
    for target_name in TARGET_COLS:
        best_cfg = best_config_by_target[target_name]
        base_params_for_target = {
            **LGBM_BASE_PARAMS,
            **{k: v for k, v in best_cfg.items() if k in LGBM_CANDIDATES[0]},
        }
        
        for seed in ENSEMBLE_SEEDS[:ENSEMBLE_SIZE]:
            params = {
                **base_params_for_target,
                'random_state': seed,
                'n_estimators': int(best_iteration_by_target[target_name]),
            }
            
            model = LGBMRegressor(**params)
            model.fit(X_train, target_arrays[target_name]['train'])
            ensemble_models[target_name].append(model)
        
        print(f"Trained {len(ensemble_models[target_name])} models for {target_name}")
    
    # Evaluate ensemble on holdout
    ensemble_holdout_results = {}
    for target_name in TARGET_COLS:
        ensemble_preds = np.column_stack([
            m.predict(X_holdout) for m in ensemble_models[target_name]
        ])
        pred_mean = np.mean(ensemble_preds, axis=1)
        ensemble_holdout_results[target_name] = reg_metrics(
            target_arrays[target_name]['holdout'], pred_mean
        )
    
    print("\nEnsemble holdout performance:")
    display(pd.DataFrame(ensemble_holdout_results).T.round(4))
    
    # Compare single vs ensemble
    comparison = pd.DataFrame({
        'single': [holdout_results[t]['Spearman'] for t in TARGET_COLS],
        'ensemble': [ensemble_holdout_results[t]['Spearman'] for t in TARGET_COLS],
    }, index=TARGET_COLS)
    comparison['improvement'] = comparison['ensemble'] - comparison['single']
    print("\nSingle vs Ensemble Spearman:")
    display(comparison.round(4))

Trained 5 models for long_quality
Trained 5 models for short_quality
Trained 5 models for signed_quality

Ensemble holdout performance:


,MAE,RMSE,R2,Spearman
long_quality,0.5008,0.7321,0.0686,0.2927
short_quality,0.4937,0.7228,0.0631,0.2847
signed_quality,0.8964,1.2239,0.0706,0.2330



Single vs Ensemble Spearman:


,single,ensemble,improvement
long_quality,0.2925,0.2927,0.0002
short_quality,0.2849,0.2847,-0.0002
signed_quality,0.2327,0.2330,0.0002


In [34]:
# Trading-style proxy analysis on the signed target
y_true_signed = target_arrays["signed_quality"]["holdout"]
y_pred_signed = signed_pred
thresholds = np.quantile(np.abs(y_pred_signed), np.linspace(0.50, 0.95, 10))

sweep_rows = []
for thr in thresholds:
    pred_sign = np.zeros(len(y_pred_signed), dtype=int)
    pred_sign[y_pred_signed >= thr] = 1
    pred_sign[y_pred_signed <= -thr] = -1
    taken = pred_sign != 0

    if taken.sum() == 0:
        sweep_rows.append({
            "threshold": float(thr),
            "coverage": 0.0,
            "taken": 0,
            "long_count": 0,
            "short_count": 0,
            "sign_hit_rate": np.nan,
            "aligned_signed_mean": np.nan,
            "taken_spearman": np.nan,
        })
        continue

    sign_hit = np.mean(np.sign(y_true_signed[taken]) == pred_sign[taken])
    aligned_signed = np.where(
        pred_sign[taken] == 1, y_true_signed[taken], -y_true_signed[taken]
    )
    sweep_rows.append({
        "threshold": float(thr),
        "coverage": float(taken.mean()),
        "taken": int(taken.sum()),
        "long_count": int((pred_sign == 1).sum()),
        "short_count": int((pred_sign == -1).sum()),
        "sign_hit_rate": float(sign_hit),
        "aligned_signed_mean": float(np.mean(aligned_signed)),
        "taken_spearman": float(safe_spearman(y_true_signed[taken], y_pred_signed[taken])),
    })

threshold_df = pd.DataFrame(sweep_rows)
print("Signed-model threshold sweep:")
display(threshold_df.round(4))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=threshold_df["threshold"],
    y=threshold_df["aligned_signed_mean"],
    mode="lines+markers",
    name="Aligned signed mean",
))
fig.add_trace(go.Scatter(
    x=threshold_df["threshold"],
    y=threshold_df["sign_hit_rate"],
    mode="lines+markers",
    name="Sign hit rate",
))
fig.update_layout(
    title="Trading Proxy vs Confidence Threshold",
    xaxis_title="Absolute prediction threshold",
    yaxis_title="Score",
    height=420,
)
fig.show()

Signed-model threshold sweep:


,threshold,coverage,taken,long_count,short_count,sign_hit_rate,aligned_signed_mean,taken_spearman
0,0.2688,0.50,25000,12175,12825,0.6127,0.4370,0.3000
1,0.2975,0.45,22500,11029,11471,0.6164,0.4525,0.3086
2,0.3276,0.40,20000,9858,10142,0.6230,0.4730,0.3187
3,0.3589,0.35,17500,8697,8803,0.6295,0.4901,0.3298
4,0.3911,0.30,15000,7563,7437,0.6375,0.5136,0.3419
5,0.4187,0.25,12500,6334,6166,0.6462,0.5405,0.3551
6,0.4487,0.20,10000,5033,4967,0.6544,0.5579,0.3725
7,0.4815,0.15,7500,3702,3798,0.6691,0.5849,0.3967
8,0.5239,0.10,5000,2355,2645,0.6994,0.6352,0.4239
9,0.5889,0.05,2500,1135,1365,0.7348,0.7055,0.4695


In [35]:
# Export the model pack
today = pd.Timestamp.now().strftime("%Y%m%d")
ds_title = DS_NAME.split("/")[-1].split(".")[0]
model_name = "_".join([ds_title, "LightGBM", today, "spearman_prod", "TP" + str(TP_MULT), "SL" + str(SL_MULT)])

model_pack_dir = Path("../ModelWorkbench/ModelPacks")
model_pack_dir.mkdir(parents=True, exist_ok=True)
model_pack_path = model_pack_dir / f"{model_name}_model.pkl"

model_info = {
    "dataset_name": ds_title,
    "dataset_dir": DS_NAME,
    "date_trained": today,
    "model_type": "LightGBMRegressionPack",
    "task": "regression",
    "primary_target": PRIMARY_TARGET,
    "targets": TARGET_COLS,
    "cv_scheme": {
        "type": "expanding_walk_forward",
        "folds": len(wf_folds),
        "gap_rows": max(WFO_GAP_ROWS, outcome_params["max_horizon"]),
        "tuning_rows": int(len(X_tune)),
        "holdout_fraction": TRAIN_FRACTION,
        "robustness_penalty": ROBUSTNESS_PENALTY,
    },
}

pack = {
    "model": final_models[PRIMARY_TARGET],
    "primary_model": final_models[PRIMARY_TARGET],
    "aux_models": {k: v for k, v in final_models.items() if k != PRIMARY_TARGET},
    "model_class": final_models[PRIMARY_TARGET].__class__,
    "model_class_source": inspect.getsource(final_models[PRIMARY_TARGET].__class__),
    "model_params": {
        target: {
            **LGBM_BASE_PARAMS,
            **{k: v for k, v in best_config_by_target[target].items() if k in LGBM_CANDIDATES[0]},
            "n_estimators": int(best_iteration_by_target[target]),
        }
        for target in TARGET_COLS
    },
    "model_info": model_info,
    "features": features,
    "selected_features": selected_features if "selected_features" in globals() else None,

    "feature_count": X_train.shape[1],
    "feature_function": FEATURES,
    "feature_function_source": inspect.getsource(FEATURES),
    "preprocess_function": preprocess_ohlcv,
    "preprocess_function_source": inspect.getsource(preprocess_ohlcv),
    "preprocess_args": preprocess_ohlcv_args,
    "scaler": scaler,
    "label_function": calculate_trade_outcomes_capped,
    "label_function_source": inspect.getsource(calculate_trade_outcomes_capped),
    "label_params": None,
    "regime_params": regime_params,
    "outcome_params": outcome_params,
    "trading_hours": None,
    "input_shape": None,
    "data_split": {
        "train_rows": len(X_train),
        "holdout_rows": len(X_holdout),
        "train_fraction": TRAIN_FRACTION,
        "max_tuning_rows": MAX_TUNING_ROWS,
    },
    "validation": {
        "cv_summary": tuning_summary.to_dict(orient="index"),
        "cv_results": {
            target: tuning_results[target].head(10).to_dict(orient="records")
            for target in TARGET_COLS
        },
        "holdout": {
            target: {k: float(v) for k, v in holdout_results[target].items()}
            for target in holdout_results
        },
        "threshold_sweep": threshold_df.to_dict(orient="records"),
    },
}

# Add classifiers to pack if trained
if TRAIN_CLASSIFIER:
    pack['classifiers'] = final_classifiers
    pack['classifier_validation'] = classifier_holdout_results

# Add ensemble models to pack if trained
if ENSEMBLE_SIZE > 1:
    pack['ensemble_models'] = ensemble_models
    pack['ensemble_size'] = ENSEMBLE_SIZE
    pack['ensemble_seeds'] = ENSEMBLE_SEEDS[:ENSEMBLE_SIZE]
    pack['ensemble_validation'] = ensemble_holdout_results

with open(model_pack_path, "wb") as fh:
    pickle.dump(pack, fh)

print(f"Saved model pack to {model_pack_path}")

Saved model pack to ../ModelWorkbench/ModelPacks/US500_M1_520weeks_LightGBM_20260704_spearman_prod_TP4.0_SL2.5_model.pkl


In [36]:
# Model importance snapshot for the primary deployment model
importance_df = pd.DataFrame({
    "feature": features,
    "importance": final_models[PRIMARY_TARGET].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

display(importance_df.head(30))

fig = go.Figure(go.Bar(
    x=importance_df.head(30)["importance"][::-1],
    y=importance_df.head(30)["feature"][::-1],
    orientation="h",
))
fig.update_layout(
    title=f"Top 30 Feature Importances - {PRIMARY_TARGET}",
    height=700,
    xaxis_title="Gain / split importance",
    yaxis_title="Feature",
)
fig.show()

,feature,importance
0,fl_atr_ratio_14,397
1,fl_close_loc,366
2,fl_hour_cos,277
3,fl_hour_sin,271
4,fl_price_loc_256,261
5,fl_hl_range_atr,247
6,fl_rv_60,201
7,fl_price_loc_128,161
8,fl_pct_from_low_60,154
9,fl_squeeze_mom,134
